In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset, random_split
import torchvision.datasets as datasets
import os
%matplotlib inline

/home/ddsukhoverkhova/.conda/envs/mc_lib_env/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [3]:
def load_data(Jd, l, num_conf, T, num_temps, batch_size, shuffle_opt, opt='train'):
    datasets = []
    for j in range(num_temps):
        path = f'data_spins/{Jd}_{opt}/spins_{l}_{T[j]}.npy'
        #if os.path.isfile(path):
        with open(path, 'rb') as f:
            x = np.load(f)   
        tensor_x = torch.Tensor(x).unsqueeze(1)
        path = f'data_spins/{Jd}_{opt}/answ_{l}_{T[j]}.npy'
        #if os.path.isfile(path):
        with open(path, 'rb') as f:
            y = np.load(f)
        tensor_y = torch.from_numpy(y).type(torch.float32)

        datasets.append(TensorDataset(tensor_x, tensor_y))


    dataset = torch.utils.data.ConcatDataset(datasets)

    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle_opt)

In [4]:
class Net(nn.Module):
    def __init__(self, l):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 64, 2)
        self.pool = nn.MaxPool2d(2, 2)
        self.act_hid = nn.ReLU()
        self.fc1 = nn.Linear(64*int(l/2-1)*int(l/2-1), 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.pool(x)
        x = self.act_hid(x)
        x = x.view(-1, 64*int(l/2-1)*int(l/2-1))
        x = self.fc1(x)
        x = self.act_hid(x)
        x = self.fc2(x)
        return x

In [5]:
def train(l, train_dataloader, num_epoch, criterion, batch_size):
    model = Net(l)
    optimizer = torch.optim.Adam(model.parameters(), lr=1.0e-4)
    act = nn.Sigmoid()

    for epoch in range(num_epoch):  
        running_loss = 0.0
        accuracy = 0.0
        pbar = tqdm(enumerate(train_dataloader), total=len(train_dataloader))
        for i, data in pbar:
            inputs, labels = data
            inputs = inputs.to(device)
            labels = labels.to(device)  
            
            model.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            #outputs = act(outputs)

            outputs = outputs.squeeze(1) # к одной размерности с labels

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            accuracy += (batch_size - sum(abs(labels - act(outputs)))).float().mean()

            pbar.set_description(
                    f"Loss: {running_loss/((i+1)*batch_size)} "
                    f"Accuracy: {accuracy * 100  / ((i+1)*batch_size)}"
            )

    print('Training completed')
    return model, optimizer

In [6]:
def testing(model, test_dataloader, criterion, batch_size):
    outp = []
    errors = []
    accuracy = 0.0
    act = nn.Sigmoid()
    with torch.no_grad():
        for i, data in enumerate(test_dataloader):
            inputs, labels = data
            inputs = inputs.to(device)
            labels = labels.to(device)  
            model.to(device)
            outputs = model(inputs)
            #outputs = act(outputs)
            outputs = outputs.squeeze(1)
            outp.append(act(outputs).item())
            loss = criterion(outputs, labels)
            errors.append(loss.item())

            accuracy += (1 - sum(abs(labels - act(outputs)))).float().mean()

    print("Accuracy = {}".format(accuracy / len(test_dataloader)))
    return outp, errors

In [7]:
roots = [2.2691853142129728, 2.104982167992544, 1.932307699120554, 1.749339162933206, 1.5536238493280832, 1.34187327905057, 1.109960313758399, 0.8541630993606272, 0.5762735442012712, 0.2885386111960936, 0.03198372863548067]
jds = [0.0, -0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, -1.0]
get_crit_T = dict(zip(jds, roots))

In [8]:
### training ###

def train_and_save(Jd, l, num_temps):
    num_conf_tr = 2048
    num_conf_ts = 512
    num_epoch = 1

    T_c = get_crit_T[Jd]
    T = np.round(np.linspace(T_c - 0.3, T_c + 0.3, num_temps), 4)

    criterion = nn.BCEWithLogitsLoss()     

    train_dataloader = load_data(Jd, l, num_conf_tr, T, num_temps, batch_size=4, shuffle_opt=True, opt='train')
    print(f'Start training for L = {l}')
    model, optimizer = train(l, train_dataloader, num_epoch, criterion, batch_size=4)

    PATH = f'models/{l}_{Jd}_{T[0]}_{T[-1]}_{num_temps}_{epoch}_epochs.pt'
    #PATH = f'models/{l}_{Jd}_{T[0]}_{T[-1]}_{num_temps}.pt'
    torch.save(model.state_dict(), PATH)
    PATH = f'models/{l}_{Jd}_{T[0]}_{T[-1]}_{num_temps}_{epoch}_epochs_opt.pt'
    #PATH = f'models/{l}_{Jd}_{T[0]}_{T[-1]}_{num_temps}.pt'
    torch.save(optimizer.state_dict(), PATH)

In [ ]:
L = [60]
Jd = 0.0
num_temps = 100
for l in L:
    train_and_save(Jd, l, num_temps)

Start training for L = 60


Loss: 0.029709162509045105 Accuracy: 93.5464859008789:  36%|███▌      | 18259/51200 [19:18<3:22:37,  2.71it/s] 

In [40]:
l = 20
Jd = -0.1
num_temps = 100
T_c = get_crit_T[Jd]
T = np.linspace(T_c - 0.3, T_c + 0.3, num_temps)
T = np.round(T, 4)
#T = np.round(np.linspace(0.03, 3.5, num_temps), 4)

if Jd == 0.0:
    opt = 'train'
    T_miss = []
    for j in range(num_temps):
            path = f'data_spins/{Jd}_{opt}/spins_{l}_{T[j]}.npy'
            if not os.path.isfile(path):
                T_miss.append(T[j])
    print(T_miss, len(T_miss))

opt = 'test'
T_miss = []
for j in range(num_temps):
        path = f'data_spins/{Jd}_{opt}/spins_{l}_{T[j]}.npy'
        if not os.path.isfile(path):
            T_miss.append(T[j])
print(T_miss, len(T_miss))

[1.805, 1.811, 1.8171, 1.8232, 1.8292, 1.8353, 1.8413, 1.8474, 1.8535, 1.8595, 1.8656, 1.8716, 1.8777, 1.8838, 1.8898, 1.8959, 1.902] 17


In [14]:
### testing ###

def get_errs_outs(Jd, l, num_temps):
    T_c = get_crit_T[Jd]
    T = np.linspace(T_c - 0.3, T_c + 0.3, num_temps)
    T = np.round(T, 4)
    
    num_conf_tr = 2048
    num_conf_ts = 512

    criterion = nn.BCEWithLogitsLoss()     
    
    print(f'Start testing for L = {l}, Jd = {Jd}')
    model = Net(l)
    T_c_ = get_crit_T[0.0]
    T_ = np.round(np.linspace(T_c_ - 0.3, T_c_ + 0.3, num_temps), 4)
    #PATH = f'models/{l}_0.0_{T_[0]}_{T_[-1]}_{num_temps}.pt'
    PATH = f'models/{l}_0.0_{T_[0]}_{T_[-1]}_{num_temps}_10_epochs.pt'

    model.load_state_dict(torch.load(PATH))
    model.eval()
    test_dataloader = load_data(Jd, l, num_conf_ts, T, num_temps, batch_size=1, shuffle_opt=False, opt='test')
    outp, errors = testing(model, test_dataloader, criterion, batch_size=1)
    return errors, outp

In [20]:
L = [10, 20, 30, 60, 80]
Jds = [0.0, -0.3, -0.5, -0.7, -0.9]
num_temps = 100
for Jd in Jds: 
    for l in L:
        errs_outs = get_errs_outs(Jd, l, num_temps)
        np.save(f'data_errors/{Jd}_{l}_{num_temps}_10_epochs.npy', errs_outs[0])
        #np.save(f'data_errors/{Jd}_{l}_{num_temps}.npy', errs_outs[0])
        np.save(f'data_outputs/{Jd}_{l}_{num_temps}_10_epochs.npy', errs_outs[1])
        #np.save(f'data_outputs/{Jd}_{l}_{num_temps}.npy', errs_outs[1])

Start testing for L = 10, Jd = 0.0
Accuracy = 0.6539118885993958
Start testing for L = 20, Jd = 0.0
Accuracy = 0.8285424709320068
Start testing for L = 30, Jd = 0.0
Accuracy = 0.8924677968025208
Start testing for L = 60, Jd = 0.0
Accuracy = 0.9527891278266907
Start testing for L = 80, Jd = 0.0
Accuracy = 0.9717230200767517
Start testing for L = 10, Jd = -0.3
Accuracy = 0.6880060434341431
Start testing for L = 20, Jd = -0.3
Accuracy = 0.8601046204566956
Start testing for L = 30, Jd = -0.3
Accuracy = 0.9104775786399841
Start testing for L = 60, Jd = -0.3
Accuracy = 0.9418126940727234
Start testing for L = 80, Jd = -0.3
Accuracy = 0.9533204436302185
Start testing for L = 10, Jd = -0.5
Accuracy = 0.7137434482574463
Start testing for L = 20, Jd = -0.5
Accuracy = 0.8630114793777466
Start testing for L = 30, Jd = -0.5
Accuracy = 0.897457480430603
Start testing for L = 60, Jd = -0.5
Accuracy = 0.9032341241836548
Start testing for L = 80, Jd = -0.5
Accuracy = 0.9089341163635254
Start testing fo

In [12]:
from torchsummary import summary

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # PyTorch v0.4.0
l = 20
model = Net(l).to(device)

summary(model, (1, l, l))

Layer (type:depth-idx)                   Output Shape              Param #
├─Conv2d: 1-1                            [-1, 64, 19, 19]          320
├─MaxPool2d: 1-2                         [-1, 64, 9, 9]            --
├─ReLU: 1-3                              [-1, 64, 9, 9]            --
├─Linear: 1-4                            [-1, 64]                  331,840
├─ReLU: 1-5                              [-1, 64]                  --
├─Linear: 1-6                            [-1, 1]                   65
Total params: 332,225
Trainable params: 332,225
Non-trainable params: 0
Total mult-adds (M): 0.42
Input size (MB): 0.00
Forward/backward pass size (MB): 0.18
Params size (MB): 1.27
Estimated Total Size (MB): 1.45


Layer (type:depth-idx)                   Output Shape              Param #
├─Conv2d: 1-1                            [-1, 64, 19, 19]          320
├─MaxPool2d: 1-2                         [-1, 64, 9, 9]            --
├─ReLU: 1-3                              [-1, 64, 9, 9]            --
├─Linear: 1-4                            [-1, 64]                  331,840
├─ReLU: 1-5                              [-1, 64]                  --
├─Linear: 1-6                            [-1, 1]                   65
Total params: 332,225
Trainable params: 332,225
Non-trainable params: 0
Total mult-adds (M): 0.42
Input size (MB): 0.00
Forward/backward pass size (MB): 0.18
Params size (MB): 1.27
Estimated Total Size (MB): 1.45